In [26]:
import re
from datetime import datetime

In [27]:
def convert_cosy_file(input_filename, output_filename):
    with open(input_filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    header = [
       "{ ************************************************************ }\n"
        f"{{ COSY Infinity Lattice Converter                              }}\n"
        f"{{ Purpose: Auto-generation of maps (SMAPS) for Twiss functions }}\n"
        f"{{ Author: Kolokolchikov S.                                     }}\n"
        f"{{ Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}         }}\n"
        f"{{ Input file: {input_filename:<48}     }}\n"
        f"{{ Output file: {output_filename:<47}     }}\n"
        "{ ************************************************************ }\n\n"
    ]

    elem_keywords = ['QUAD', 'SBEND', 'DL', 'MH', 'RF', 'MS', 'MQ']
    global_idx = 0
    section_counts = []
    current_section_count = 0
    
    processed_lines = []
    
    for line in lines:
        # Update PROCEDURE header
        if 'PROCEDURE LATTICE' in line.upper() and 'MAPARR' not in line:
            line = re.sub(r'(PROCEDURE\s+LATTICE.*?)(?=\s*;)', r'\1 MAPARR SPNRARR', line, flags=re.IGNORECASE)

        # Handle SECTION HEADERS (keeping Indent)
        section_match = re.match(r'^([ \t]*)(\{={2,}.*?={2,}.*?\})', line)
        if section_match:
            if current_section_count > 0:
                section_counts.append(current_section_count)
            current_section_count = 0
            
            indent = section_match.group(1)
            raw_title = section_match.group(2)
            
            # Extract clean name (letters only)
            name_match = re.search(r'={2,}([A-Za-z\s]+)', raw_title)
            name = name_match.group(1).strip() if name_match else "SECTION"
            
            # Save line with placeholder for count
            processed_lines.append(f"{indent}{{========{name}========== elements: REPLACE_ME }}\n")
            continue

        # Handle ELEMENTS (keeping Indent)
        elem_match = re.match(r'^([ \t]*)(' + '|'.join(elem_keywords) + r')\b([^;]+;)(.*)', line, re.IGNORECASE)
        if elem_match:
            global_idx += 1
            current_section_count += 1
            
            indent = elem_match.group(1)
            elem_cmd = elem_match.group(2) + elem_match.group(3) # Code until ;
            comment = elem_match.group(4).strip() # Everything after ;
            
            new_line = f"{indent}UM; {elem_cmd} {comment} SMAPS {global_idx} MAPARR SPNRARR;\n"
            processed_lines.append(new_line)
        else:
            processed_lines.append(line)

    # Add count for the last section
    if current_section_count > 0:
        section_counts.append(current_section_count)

    # Combine everything
    final_content = "".join(header) + "".join(processed_lines)

    # Replace placeholders with actual counts
    for count in section_counts:
        final_content = final_content.replace("REPLACE_ME", str(count), 1)

    # Update global footer
    if section_counts:
        total_str = " + ".join(map(str, section_counts))
        final_content = re.sub(r'\{elem-counting:.*?\}', f"{{elem-counting: {total_str} = {global_idx}}}", final_content)

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(final_content)
    
    print(f"Success! Output: {output_filename}")

# convert_cosy_file('input.fox', 'output.fox')

In [28]:
convert_cosy_file('magnetic.fox', 'magnetic_maps.fox')

Success! Output: magnetic_maps.fox
